[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S15/2_Practica/para_casa/S15_01_seguimiento_experimentos.ipynb)

# S15 · 01 — Seguimiento de experimentos y reproducibilidad

Sprint 15 · Proyecto: modelado y experimentos (píldora de apoyo)

**Contexto TurisData.** En el proyecto final vuestro equipo probará muchos modelos y muchos ajustes. Al tercer día nadie recuerda *"¿aquel 0,78 era con la semilla 1 o la 42? ¿con o sin la variable de antelación?"*. La rúbrica (criterio C) premia el **registro de experimentos**, y un modelo que no se puede reproducir no se puede defender. Aquí montamos un sistema de registro **muy simple** (un CSV) que funciona en cualquier equipo. Usamos las reservas de TurisData como datos de práctica; en el proyecto usaréis los vuestros.

**Este cuaderno tiene 5 ejercicios.** Los ejercicios 1, 2 y 3 son los mismos que los del cuaderno «en clase» (allí, del 1 al 3).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os, sys, time, json, random, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

sns.set_theme(style="whitegrid")
print("Listo ✔")

### ✏️ A2 · EJERCICIO 1 de 5 — Fijar las semillas

**Tu turno:** Escribe `fijar_semillas(semilla)` que fije la semilla de `random` y de `numpy` (`np.random.seed`), guarde `PYTHONHASHSEED` en `os.environ` (como texto) y **devuelva** un generador `np.random.default_rng(semilla)`. (Con TensorFlow añadirías `tf.random.set_seed(semilla)`.)

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A2).

1. ¿Por qué hay que fijar una semilla en cada biblioteca?
2. ¿Por qué `PYTHONHASHSEED` se guarda con `str(semilla)`?
3. ¿Qué devuelve `fijar_semillas`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A2, **después de intentarlo**.

In [ ]:
def fijar_semillas(semilla):
    # pista: cada biblioteca con azar tiene su propia semilla; la variable de entorno debe ser un str; default_rng crea el generador moderno de numpy
    ...

SEMILLA = 42
rng = fijar_semillas(SEMILLA)
a = (random.random(), np.random.rand(), rng.random())
rng = fijar_semillas(SEMILLA)
b = (random.random(), np.random.rand(), rng.random())
assert a == b, "Al llamar dos veces a fijar_semillas(42) deberías obtener la misma secuencia aleatoria en random, np.random y el rng devuelto"
assert os.environ.get("PYTHONHASHSEED") == "42", "PYTHONHASHSEED debe guardarse como texto '42'"
print("✔ Ejercicio 1 correcto")

### 👀 A3 · EJEMPLO — La ficha del entorno

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Por qué anotar las versiones de las librerías?
2. ¿Dónde conviene copiar esta ficha?

In [ ]:
# Ficha del entorno: cópiala en la memoria técnica y en el README del equipo
ENTORNO = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__}
print(json.dumps(ENTORNO, indent=2))

### ✏️ A4 · EJERCICIO 2 de 5 — Escribir una fila en el registro

**Tu turno:** Completa `registrar(...)`, que añade **una fila** al CSV `RUTA_REGISTRO`: si el archivo no existe, lo crea con la cabecera; si existe, añade la fila sin repetir la cabecera. El `id` es el número de filas ya existentes + 1. Devuelve el `id` asignado.

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Cómo se calcula el `id` de cada fila?
2. ¿Qué hace `header=not existe`?
3. ¿Qué ventajas tiene un CSV como registro?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
RUTA_REGISTRO = "experimentos/registro.csv"
COLUMNAS = ["id", "fecha", "responsable", "modelo", "hiperparametros", "datos_hash", "semilla",
            "metrica", "valor_cv", "desv_cv", "tiempo_s", "notas"]
os.makedirs("experimentos", exist_ok=True)
if os.path.exists(RUTA_REGISTRO):
    os.remove(RUTA_REGISTRO)          # empezamos el laboratorio con el registro vacío

def registrar(responsable, modelo, hiperparametros, datos_hash, semilla, metrica, valor_cv, desv_cv, tiempo_s, notas=""):
    # pista: cuenta las filas con pd.read_csv si el archivo existe; crea un DataFrame de una fila y guárdalo con to_csv en modo "a" (header solo si es nuevo)
    ...

id1 = registrar("prueba", "ModeloDePrueba", {"a": 1}, "abc123", 42, "AUC", 0.5, 0.01, 0.1, "fila de prueba")
id2 = registrar("prueba", "ModeloDePrueba", {"a": 2}, "abc123", 42, "AUC", 0.6, 0.01, 0.1)
tabla_prueba = pd.read_csv(RUTA_REGISTRO)
assert (id1, id2) == (1, 2), "Los id deben ser 1, 2, ... (filas existentes + 1)"
assert list(tabla_prueba.columns) == COLUMNAS and len(tabla_prueba) == 2, "El CSV debe tener la cabecera una sola vez y 2 filas"
os.remove(RUTA_REGISTRO)              # borramos las filas de prueba
print("✔ Ejercicio 2 correcto")

### 👀 A5 · EJEMPLO — Preparar los datos y la función experimento (baseline y regresión logística)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Qué AUC tiene el baseline y qué significa?
2. ¿Qué AUC medio logra la regresión logística y con qué desviación?
3. ¿Para qué sirve la huella de los datos (`DATOS_HASH`)?

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
reservas["mes_llegada"] = reservas["fecha_llegada"].dt.month
X = reservas.drop(columns=["id_reserva", "fecha_reserva", "fecha_llegada", "cancelada"])
y = reservas["cancelada"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)

# huella de los datos: si alguien cambia el CSV, el hash cambia y lo sabremos
DATOS_HASH = hashlib.md5(pd.util.hash_pandas_object(reservas, index=True).values.tobytes()).hexdigest()[:8]
print("Huella de los datos:", DATOS_HASH, "· entrenamiento:", X_ent.shape, "· test (sin tocar):", X_test.shape)

categoricas = X.select_dtypes("object").columns.tolist()
def crear_pipeline(estimador, columnas):
    numericas = [c for c in columnas if c not in categoricas]
    prep = ColumnTransformer([("num", StandardScaler(), numericas), ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas)])
    return Pipeline([("prep", prep), ("m", estimador)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

def experimento(nombre, estimador, hiperparametros, responsable="equipo", notas="", Xd=None):
    """Evalúa con CV (AUC), mide el tiempo y registra la fila. Devuelve (media, desviación)."""
    Xd = X_ent if Xd is None else Xd
    t0 = time.time()
    puntuaciones = cross_val_score(crear_pipeline(estimador, Xd.columns), Xd, y_ent, cv=cv, scoring="roc_auc")
    registrar(responsable, nombre, hiperparametros, DATOS_HASH, SEMILLA, "AUC", puntuaciones.mean(), puntuaciones.std(), time.time() - t0, notas)
    return puntuaciones.mean(), puntuaciones.std()

print("Baseline (predice la clase más frecuente):", experimento("Baseline", DummyClassifier(strategy="prior"), {"strategy": "prior"}, notas="referencia mínima"))
print("Regresión logística:", experimento("RegresionLogistica", LogisticRegression(max_iter=2000), {"C": 1.0}))

### ✏️ A6 · EJERCICIO 3 de 5 — Registrar bosque y gradient boosting y elegir candidato

**Tu turno:** Registra dos experimentos más con la función `experimento(nombre, estimador, hiperparametros)`: un **bosque aleatorio** `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2)` con nombre `"RandomForest"` y un **gradient boosting** `HistGradientBoostingClassifier(random_state=SEMILLA)` con nombre `"HistGradientBoosting"` (hiperparámetros: los que uses, en un diccionario). Después lee el registro en `tabla` (`pd.read_csv(RUTA_REGISTRO)`) y guarda en `candidato` el **nombre del modelo con mayor `valor_cv`**.

**Debe salir:** `candidato provisional: RegresionLogistica`, una tabla con 4 filas y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Qué modelo sale como candidato provisional?
2. ¿Qué AUC medio tienen el bosque y el gradient boosting?
3. ¿Qué más, además del AUC, se mira para elegir?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
# pista: llama a experimento dos veces (RandomForest y HistGradientBoosting); lee el CSV con RUTA_REGISTRO; idxmax sobre valor_cv te da la fila del mejor
tabla = ...
candidato = ...

assert len(tabla) == 4, "El registro debería tener 4 filas: baseline, logística, bosque y gradient boosting"
assert set(["RandomForest", "HistGradientBoosting"]) <= set(tabla["modelo"]), "Registra los dos modelos con los nombres indicados"
assert candidato == tabla.sort_values("valor_cv").iloc[-1]["modelo"], "candidato debe ser el modelo con mayor valor_cv"
print("candidato provisional:", candidato)
tabla[["id", "modelo", "hiperparametros", "valor_cv", "desv_cv", "tiempo_s"]]
print("✔ Ejercicio 3 correcto")

### 👀 A7 · EJEMPLO — Comparativa de modelos en validación cruzada

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Qué modelo tiene la barra más larga?
2. ¿Qué indica la barra de error de cada modelo?
3. Si dos modelos difieren menos que su desviación, ¿cuál eliges?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
t = tabla.sort_values("valor_cv")
ax.barh(t["modelo"], t["valor_cv"], xerr=t["desv_cv"], color="tab:blue", capsize=4)
ax.axvline(0.5, color="gray", linestyle="--", label="Azar (AUC 0,5)")
ax.set_xlim(0.4, 0.85); ax.set_xlabel("AUC medio en validación cruzada (± desviación)")
ax.set_title("Comparativa de modelos (validación cruzada, sin usar el test)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

### ✏️ A8 · EJERCICIO 4 de 5 — Probar tres profundidades del bosque

**Tu turno:** Prueba un bosque aleatorio con `max_depth` = 3, 6 y 10 (resto de parámetros como antes), **registrando cada prueba** con `experimento` y el nombre `f"RandomForest_prof{d}"`. Guarda en `resultados_prof` un diccionario `{profundidad: AUC medio}` y en `mejor_prof` la profundidad con mejor AUC.

**Debe salir:** `mejor profundidad: 6 {3: 0.742, 6: 0.745, 10: 0.742}` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A8).

1. ¿Qué profundidad da el mejor AUC y cuál es su valor?
2. ¿Es una diferencia real respecto a 3 y a 10?
3. ¿Por qué cada prueba se registra con un nombre distinto?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A8, **después de intentarlo**.

In [ ]:
# pista: un bucle sobre las tres profundidades; experimento devuelve (media, desviación)
resultados_prof = ...
mejor_prof = ...

assert set(resultados_prof) == {3, 6, 10}, "resultados_prof debe tener las claves 3, 6 y 10"
assert len(pd.read_csv(RUTA_REGISTRO)) == 7, "Deberías tener 7 filas en el registro (4 anteriores + 3 nuevas)"
assert mejor_prof in (3, 6, 10)
print("mejor profundidad:", mejor_prof, {k: round(v, 3) for k, v in resultados_prof.items()})
print("✔ Ejercicio 4 correcto")

### 👀 A9 · EJEMPLO — Una variable con fuga de datos

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A9).

1. ¿Qué AUC da el modelo con la variable con fuga y cuál sin ella?
2. ¿Por qué `reembolso_emitido` es una fuga?
3. ¿Qué pregunta ayuda a detectar fugas en cada variable?

In [ ]:
X_fuga = X_ent.copy()
X_fuga["reembolso_emitido"] = ((y_ent == 1) & (rng.random(len(y_ent)) < 0.9)).astype(int)
m, d = experimento("LogisticaConFuga", LogisticRegression(max_iter=2000), {"C": 1.0}, notas="ALARMA: AUC sospechosamente alto (fuga: reembolso_emitido)", Xd=X_fuga)
mejor_sin_fuga = pd.read_csv(RUTA_REGISTRO).query("modelo != 'LogisticaConFuga'")["valor_cv"].max()
print(f"AUC con la variable con fuga: {m:.3f} (frente a {mejor_sin_fuga:.3f} del mejor modelo sin ella) → ¡descártala y anótalo en el registro!")

### ✏️ A10 · EJERCICIO 5 de 5 — Evaluar el candidato en el test una sola vez

**Tu turno:** Elige el candidato de la tabla (**solo con criterios de validación cruzada, sin fuga**): el `RandomForest_prof{mejor_prof}` o el ganador del ejercicio 3, el que tenga mayor `valor_cv` entre las filas **sin** la palabra "Fuga" en el nombre. Reentrénalo con todo `X_ent` y calcula el AUC en `X_test` (`auc_test`). Guarda en `auc_cv_candidato` su AUC de validación cruzada para compararlos. *(Pista: si el test sale muy por debajo de la validación cruzada, hay sobreajuste o fuga.)*

**Debe salir:** `candidato: RegresionLogistica · AUC validación 0.755 · AUC test 0.745` y ✔ Ejercicio 5 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A10).

1. ¿Qué candidato se elige y qué AUC tiene en validación y en test?
2. ¿Por qué se excluye el modelo con fuga?
3. ¿Qué pasaría si se elige el modelo mirando el test varias veces?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A10, **después de intentarlo**.

In [ ]:
# pista: filtra las filas de la tabla sin "Fuga"; según el nombre elegido, construye el estimador con los mismos hiperparámetros; predict_proba(X_test)[:, 1]
tabla = ...
fila = ...
auc_test = ...
auc_cv_candidato = ...

assert 0.6 < auc_test < 0.85, "El AUC en test debería ser realista (entre 0,6 y 0,85); revisa que hayas excluido el modelo con fuga"
assert abs(auc_test - auc_cv_candidato) < 0.06, "Test y validación cruzada deberían ser parecidos; si no, sospecha de fuga o sobreajuste"
print(f"candidato: {fila['modelo']} · AUC validación {auc_cv_candidato:.3f} · AUC test {auc_test:.3f}")
print("✔ Ejercicio 5 correcto")

### 👀 A11 · EJEMPLO — MLflow en local (opcional, requiere internet)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A11).

1. ¿Es obligatorio usar MLflow en el proyecto?
2. ¿Qué ventaja tiene MLflow frente al CSV?

In [ ]:
USAR_MLFLOW = False        # OPCIONAL: cámbialo a True solo si tu equipo quiere probar MLflow (necesita internet para instalarlo)

if USAR_MLFLOW:
    # %pip install -q mlflow       # <- descomenta en Colab; en local: pip install mlflow
    import mlflow
    try:
        import sqlalchemy                                  # instalado con MLflow completo: usamos la base de datos local
        URI = "sqlite:///mlflow.db"
    except ImportError:                                    # instalación ligera: carpeta local mlruns/
        os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
        URI = "file:./mlruns"
    mlflow.set_tracking_uri(URI)
    mlflow.set_experiment("turisdata_cancelaciones")
    with mlflow.start_run(run_name="RandomForest_demo"):
        mlflow.log_params({"n_estimators": 200, "min_samples_leaf": 5, "semilla": SEMILLA})
        mlflow.log_metric("auc_cv", float(fila["valor_cv"]))
        mlflow.log_metric("auc_test", float(auc_test))
        mlflow.set_tag("datos_hash", DATOS_HASH)
    print(f"Registrado en {URI}. Para verlo, en una terminal:  mlflow ui --backend-store-uri {URI}  y visita http://127.0.0.1:5000")
else:
    print("MLflow desactivado (opcional). El registro en CSV ya cumple lo que pide la rúbrica.")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

**Contexto.** En el proyecto final tu equipo probará muchos modelos y muchos ajustes. Al tercer día nadie recuerda «¿aquel 0,78 era con la semilla 1 o la 42? ¿con o sin la variable de antelación?». La rúbrica (criterio C) premia el **registro de experimentos**, y un modelo que no se puede reproducir no se puede defender. Aquí montamos un sistema de registro **muy simple** (un CSV) que funciona en cualquier equipo, con las reservas de TurisData como datos de práctica.

- **Qué hace:**
    - Carga las herramientas generales (`os`, `sys`, `time`, `json`, `random`, `hashlib`, `datetime`), `numpy`, `pandas`, `matplotlib`, `seaborn` y, de `scikit-learn`, la validación cruzada, el preprocesado, los modelos (baseline, regresión logística, bosque aleatorio y gradient boosting) y el AUC.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Reproducibilidad: semillas y versiones

#### A2 · Ejercicio 1 — Fijar las semillas

**Idea.** Muchos pasos usan azar (separar datos, inicializar pesos, submuestrear). Si no fijas la
**semilla**, cada ejecución da un resultado ligeramente distinto. Además, distintas **versiones** de las
librerías pueden cambiar resultados. Regla del equipo: **una función `fijar_semillas` al inicio de cada
notebook** y las versiones anotadas en el registro.

- **Qué hace:**
    - `random.seed(semilla)` y `np.random.seed(semilla)` fijan la semilla de cada biblioteca con azar.
    - `os.environ["PYTHONHASHSEED"] = str(semilla)` guarda la semilla como **texto** en una variable de entorno (Python solo admite cadenas ahí).
    - `np.random.default_rng(semilla)` crea el generador moderno de NumPy y se **devuelve**.
    - La comprobación llama dos veces a la función con 42 y verifica que `random`, `np.random` y el generador devuelven los mismos números.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto` (la función no imprime nada).
- **Cómo leerlo:**
    - Cada librería tiene su propio generador: hay que fijarlos **todos**. Con TensorFlow se añadiría `tf.random.set_seed(semilla)`. Una función al inicio del cuaderno basta para que otra persona obtenga *exactamente* tus resultados.
- **Si falla:**
    - `Al llamar dos veces a fijar_semillas(42) deberías obtener la misma secuencia…` → falta fijar alguna de las tres semillas (`random`, `np.random`) o no devuelves `default_rng(semilla)`.
    - `PYTHONHASHSEED debe guardarse como texto '42'` → has guardado el número, no el texto (usa `str(semilla)`).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué hay que fijar una semilla en cada biblioteca?* → Porque `random`, NumPy y los modelos tienen **generadores independientes**: fijar uno no fija los demás.
2. *¿Por qué `PYTHONHASHSEED` se guarda con `str(semilla)`?* → Porque las variables de entorno solo admiten **texto**.
3. *¿Qué devuelve `fijar_semillas`?* → Un **generador** `np.random.default_rng(semilla)` para usarlo después.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def fijar_semillas(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    os.environ["PYTHONHASHSEED"] = str(semilla)
    return np.random.default_rng(semilla)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A3 · Ejemplo — La ficha del entorno

- **Qué hace:**
    - `ENTORNO` guarda las versiones de Python, NumPy, pandas y scikit-learn; `json.dumps(..., indent=2)` las imprime con sangría.
- **Qué debe salir:**
    - Un diccionario con cuatro claves: `python`, `numpy`, `pandas` y `scikit-learn`, cada una con su versión (las de tu entorno; por ejemplo `"scikit-learn": "1.9.1"`).
- **Cómo leerlo:**
    - Distintas versiones de una librería pueden cambiar los resultados. Copia esta ficha en la memoria técnica y en el README del equipo.
- **Si falla:**
    - `NameError: name 'sklearn' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué anotar las versiones de las librerías?* → Porque **otra versión puede dar otros resultados**: sin ellas, no se puede reproducir con certeza.
2. *¿Dónde conviene copiar esta ficha?* → En la **memoria técnica** y en el **README** del equipo.

</details>

## 2. El registro de experimentos: una fila por prueba

#### A4 · Ejercicio 2 — Escribir una fila en el registro

**Idea.** Cada vez que entrenas y evalúas, anotas una fila: **cuándo, qué modelo, con qué ajustes, con
qué datos, qué métrica y cuánto tardó**. Un CSV en el repositorio del equipo (`experimentos/registro.csv`)
se puede abrir con Excel, se versiona con Git y no requiere instalar nada.

Columnas mínimas: `id`, `fecha`, `responsable`, `modelo`, `hiperparametros` (JSON), `datos_hash`,
`semilla`, `metrica`, `valor_cv`, `desv_cv`, `tiempo_s`, `notas`.

- **Qué hace:**
    - Las primeras líneas (ya hechas) definen la ruta del registro, la lista de columnas, crean la carpeta `experimentos` y **vacían el registro** para empezar de cero.
    - `registrar(...)` mira si el CSV existe (`existe`) y cuenta sus filas (`n`); el `id` es `n + 1`.
    - Crea un `DataFrame` de una fila con todos los campos (la fecha actual, los hiperparámetros como JSON, los números redondeados) y lo añade con `to_csv(..., mode="a", header=not existe, index=False)`: añade sin repetir la cabecera.
    - La comprobación registra dos filas de prueba, verifica los `id` y las columnas, y borra el archivo.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto` (la función no imprime nada).
- **Cómo leerlo:**
    - El registro es un **CSV**: se abre con Excel, se versiona con Git y no necesita instalar nada. Cada llamada añade una fila con cuándo, quién, qué modelo, con qué ajustes y datos, qué métrica y cuánto tardó.
- **Si falla:**
    - `Los id deben ser 1, 2, ... (filas existentes + 1)` → no cuentas las filas existentes o devuelves otro valor.
    - `El CSV debe tener la cabecera una sola vez y 2 filas` → escribes la cabecera en cada llamada (`header` debe ser `True` solo si el archivo es nuevo) o no usas el modo `"a"` (añadir).
    - `NameError: name 'datetime' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cómo se calcula el `id` de cada fila?* → **Filas ya existentes + 1**.
2. *¿Qué hace `header=not existe`?* → Escribe la cabecera **solo la primera vez** (cuando el archivo no existía).
3. *¿Qué ventajas tiene un CSV como registro?* → Se abre con **Excel**, se **versiona con Git** y no requiere instalar nada.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
RUTA_REGISTRO = "experimentos/registro.csv"
COLUMNAS = ["id", "fecha", "responsable", "modelo", "hiperparametros", "datos_hash", "semilla",
            "metrica", "valor_cv", "desv_cv", "tiempo_s", "notas"]
os.makedirs("experimentos", exist_ok=True)
if os.path.exists(RUTA_REGISTRO):
    os.remove(RUTA_REGISTRO)          # empezamos el laboratorio con el registro vacío

def registrar(responsable, modelo, hiperparametros, datos_hash, semilla, metrica, valor_cv, desv_cv, tiempo_s, notas=""):
    existe = os.path.exists(RUTA_REGISTRO)
    n = len(pd.read_csv(RUTA_REGISTRO)) if existe else 0
    fila = pd.DataFrame([{
        "id": n + 1, "fecha": datetime.now().strftime("%Y-%m-%d %H:%M"), "responsable": responsable, "modelo": modelo,
        "hiperparametros": json.dumps(hiperparametros, sort_keys=True), "datos_hash": datos_hash, "semilla": semilla,
        "metrica": metrica, "valor_cv": round(float(valor_cv), 4), "desv_cv": round(float(desv_cv), 4),
        "tiempo_s": round(float(tiempo_s), 2), "notas": notas}], columns=COLUMNAS)
    fila.to_csv(RUTA_REGISTRO, mode="a", header=not existe, index=False)
    return n + 1
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Baseline y modelos con validación cruzada

#### A5 · Ejemplo — Preparar los datos y la función experimento (baseline y regresión logística)

**Idea.** Para comparar modelos se usa **validación cruzada**: se parte el conjunto de entrenamiento en 5
trozos y se entrena 5 veces, evaluando cada vez en el trozo que no se ha visto. Obtenemos una **media**
y una **desviación** (¡importante para saber si una diferencia es real o ruido!). El **test** se
reserva para el final y se usa **una sola vez**. Con series temporales, la partición debe respetar el orden
del tiempo (`TimeSeriesSplit`), nunca al azar.

**Ejemplo resuelto:** preparamos datos y una función que entrena, evalúa y **registra automáticamente**.

- **Qué hace:**
    - Carga las reservas, separa `X` e `y` y guarda el 20 % para el test (`X_test`, que **no se toca** hasta el final).
    - `DATOS_HASH` es la huella de los datos: si alguien cambia el CSV, cambia.
    - `crear_pipeline` prepara los datos y el modelo en una sola pieza; `cv` es la validación cruzada de 5 trozos.
    - `experimento(...)` evalúa un modelo con validación cruzada (AUC), mide el tiempo, **registra la fila** con `registrar` y devuelve la media y la desviación.
    - Se prueban el baseline (predice la clase más frecuente) y la regresión logística.
- **Qué debe salir:**
    - `Huella de los datos: 800f40b2 · entrenamiento: (4800, 15) · test (sin tocar): (1200, 15)`, `Baseline (predice la clase más frecuente): (0.5, 0.0)` y `Regresión logística: (0.755, 0.0124)` (según la versión de NumPy pueden aparecer como `np.float64(0.5)`).
- **Cómo leerlo:**
    - El **baseline** da AUC 0,5 (el azar): es la referencia mínima. La logística consigue **0,755 ± 0,012**. La desviación importa: dice cuánto cambia el resultado según el trozo.
- **Si falla:**
    - `NameError: name 'registrar' is not defined` → falta el ejercicio 2.
    - `NameError: name 'SEMILLA' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC tiene el baseline y qué significa?* → **0,5**: es el azar, la referencia mínima que cualquier modelo debe superar.
2. *¿Qué AUC medio logra la regresión logística y con qué desviación?* → **0,755** ± **0,012**.
3. *¿Para qué sirve la huella de los datos (`DATOS_HASH`)?* → Para saber si **alguien ha cambiado el CSV**: si cambia, la huella cambia.

</details>

#### A6 · Ejercicio 3 — Registrar bosque y gradient boosting y elegir candidato

Registra dos experimentos más con la función `experimento(nombre, estimador, hiperparametros)`:
un **bosque aleatorio** `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2)`
con nombre `"RandomForest"` y un **gradient boosting** `HistGradientBoostingClassifier(random_state=SEMILLA)` con
nombre `"HistGradientBoosting"` (hiperparámetros: los que uses, en un diccionario). Después lee el registro
en `tabla` (`pd.read_csv(RUTA_REGISTRO)`) y guarda en `candidato` el **nombre del modelo con mayor `valor_cv`**.

- **Qué hace:**
    - `experimento("RandomForest", RandomForestClassifier(...), {...})` y `experimento("HistGradientBoosting", HistGradientBoostingClassifier(...), {...})` evalúan los dos modelos con validación cruzada y **registran** una fila cada uno.
    - `pd.read_csv(RUTA_REGISTRO)` lee el registro (`tabla`).
    - `tabla.loc[tabla["valor_cv"].idxmax(), "modelo"]` es el nombre del modelo con mayor AUC medio (`candidato`).
- **Qué debe salir:**
    - `candidato provisional: RegresionLogistica`, una tabla con 4 filas (`Baseline` 0.5000, `RegresionLogistica` 0.7550, `RandomForest` 0.7393 y `HistGradientBoosting` 0.7275) y `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - La logística gana, con el bosque y el gradient boosting algo por debajo. Pero fíjate en las desviaciones (0,011–0,013): las diferencias entre modelos son pequeñas. Y el bosque tarda unos 3,6 s frente a 0,14 s de la logística: el tiempo también es un criterio.
- **Si falla:**
    - `El registro debería tener 4 filas: baseline, logística, bosque y gradient boosting` → falta registrar alguno de los dos modelos (o los has registrado dos veces).
    - `Registra los dos modelos con los nombres indicados` → los nombres deben ser exactamente `"RandomForest"` y `"HistGradientBoosting"`.
    - `candidato debe ser el modelo con mayor valor_cv` → no has usado `idxmax` sobre `valor_cv`.
    - `NameError: name 'experimento' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué modelo sale como candidato provisional?* → **RegresionLogistica**, con AUC medio **0,755**.
2. *¿Qué AUC medio tienen el bosque y el gradient boosting?* → **0,739** y **0,728**.
3. *¿Qué más, además del AUC, se mira para elegir?* → La **desviación** (si la diferencia es real) y el **tiempo** de cada modelo.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
experimento("RandomForest", RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2),
            {"n_estimators": 200, "min_samples_leaf": 5})
experimento("HistGradientBoosting", HistGradientBoostingClassifier(random_state=SEMILLA), {"defecto": True})
tabla = pd.read_csv(RUTA_REGISTRO)
candidato = tabla.loc[tabla["valor_cv"].idxmax(), "modelo"]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A7 · Ejemplo — Comparativa de modelos en validación cruzada

- **Qué hace:**
    - Ordena la tabla por `valor_cv` y dibuja barras horizontales con el AUC medio de cada modelo y su desviación como barra de error (`xerr`).
    - La línea discontinua marca el azar (AUC 0,5).
- **Qué debe salir:**
    - Un gráfico «Comparativa de modelos (validación cruzada, sin usar el test)» con cuatro barras: `Baseline` en 0,5 y las demás entre 0,73 y 0,76, con barras de error pequeñas.
- **Cómo leerlo:**
    - si dos modelos difieren menos que su desviación, **no hay diferencia real**.
En ese caso elige el más simple, más rápido o más fácil de explicar. Anótalo en la columna `notas`.
- **Si falla:**
    - `NameError: name 'tabla' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué modelo tiene la barra más larga?* → La **regresión logística** (AUC ≈ 0,755).
2. *¿Qué indica la barra de error de cada modelo?* → La **desviación** de la validación cruzada: cuánto varía el resultado entre trozos.
3. *Si dos modelos difieren menos que su desviación, ¿cuál eliges?* → El **más simple, rápido o fácil de explicar**: no hay diferencia real.

</details>

## 4. Ajuste de hiperparámetros, también registrado

#### A8 · Ejercicio 4 — Probar tres profundidades del bosque

- **Qué hace:**
    - El bucle `for d in (3, 6, 10)` entrena y **registra** un bosque con cada `max_depth` con `experimento(f"RandomForest_prof{d}", ...)`; el nombre incluye la profundidad.
    - `resultados_prof[d] = media` guarda el AUC medio de cada profundidad en un diccionario.
    - `max(resultados_prof, key=resultados_prof.get)` da la profundidad con mejor AUC (`mejor_prof`).
- **Qué debe salir:**
    - `mejor profundidad: 6 {3: 0.742, 6: 0.745, 10: 0.742}` y `✔ Ejercicio 4 correcto`.
- **Cómo leerlo:**
    - La mejor profundidad es 6, pero las tres dan casi lo mismo (0,742 a 0,745), menos que la desviación de la validación cruzada (≈ 0,013): **no hay diferencia real**. Habría que elegir por simplicidad o velocidad.
- **Si falla:**
    - `resultados_prof debe tener las claves 3, 6 y 10` → el diccionario no tiene esas tres profundidades como claves.
    - `Deberías tener 7 filas en el registro (4 anteriores + 3 nuevas)` → no registras las tres pruebas con `experimento`, o has ejecutado la celda varias veces (se añaden filas repetidas).
    - `NameError: name 'experimento' is not defined` → falta el ejemplo de la validación cruzada.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué profundidad da el mejor AUC y cuál es su valor?* → **6**, con **0,745**.
2. *¿Es una diferencia real respecto a 3 y a 10?* → **No**: 0,742 frente a 0,745 es menos que la desviación (≈ 0,013).
3. *¿Por qué cada prueba se registra con un nombre distinto?* → Para poder **distinguir las filas** del registro y saber qué ajuste dio cada resultado.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
resultados_prof = {}
for d in (3, 6, 10):
    media, desv = experimento(f"RandomForest_prof{d}",
                              RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_depth=d, random_state=SEMILLA, n_jobs=2),
                              {"n_estimators": 200, "min_samples_leaf": 5, "max_depth": d}, notas="ajuste de profundidad")
    resultados_prof[d] = media
mejor_prof = max(resultados_prof, key=resultados_prof.get)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. Alarma: una métrica demasiado buena (fuga de datos)

#### A9 · Ejemplo — Una variable con fuga de datos

**Idea.** Si tu modelo obtiene un resultado espectacular (AUC 0,99 en un problema difícil), **sospecha**.
Casi siempre hay **fuga de datos**: una variable que incluye información del futuro o del propio
objetivo. Ejemplo: una columna `reembolso_emitido` solo existe **después** de cancelar. El modelo
"acierta" pero es inútil en producción, porque en el momento de decidir esa columna aún no existe.

**Ejemplo resuelto:** la registramos con una nota de alarma y la comparamos.

**Cómo prevenir fugas:** (1) para cada variable pregunta *"¿la conocería en el momento de decidir?"*;
(2) haz el **preprocesado dentro de un `Pipeline`** (así el escalado se aprende solo con el entrenamiento);
(3) en series, parte por fechas; (4) sospecha de lo "demasiado bueno".

- **Qué hace:**
    - `X_fuga` es una copia de `X_ent` con una columna nueva, `reembolso_emitido`, que vale 1 en el 90 % de las reservas canceladas: solo existe **después** de cancelar, así que contiene la respuesta.
    - `experimento("LogisticaConFuga", ..., Xd=X_fuga)` evalúa la logística con esa columna y la registra con una **nota de alarma**.
    - `mejor_sin_fuga` es el mejor AUC de los modelos del registro **sin** fuga; se imprimen ambos.
- **Qué debe salir:**
    - `AUC con la variable con fuga: 0.972 (frente a 0.755 del mejor modelo sin ella) → ¡descártala y anótalo en el registro!`
- **Cómo leerlo:**
    - Un AUC de **0,972** en un problema donde lo normal ronda 0,75 es **demasiado bueno**: sospecha. En producción esa columna aún no existe en el momento de decidir, así que el modelo «acierta» pero es inútil.
- **Si falla:**
    - `NameError: name 'rng' is not defined` → falta el ejercicio 1.
    - `NameError: name 'experimento' is not defined` → falta el ejemplo de la validación cruzada.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC da el modelo con la variable con fuga y cuál sin ella?* → **0,972** frente a **0,755**.
2. *¿Por qué `reembolso_emitido` es una fuga?* → Porque solo existe **después** de cancelar: contiene la respuesta que se quiere predecir.
3. *¿Qué pregunta ayuda a detectar fugas en cada variable?* → **«¿La conocería en el momento de decidir?»**

</details>

## 6. Evaluar el candidato en el test, una sola vez

#### A10 · Ejercicio 5 — Evaluar el candidato en el test una sola vez

- **Qué hace:**
    - `tabla[~tabla["modelo"].str.contains("Fuga")]` deja solo los modelos legítimos; `idxmax` sobre `valor_cv` elige la fila con mejor AUC de validación cruzada (`fila`).
    - Con `json.loads(fila["hiperparametros"])` se recuperan los ajustes del registro y se construye el estimador correspondiente.
    - `crear_pipeline(estimador, X_ent.columns).fit(X_ent, y_ent)` lo reentrena con **todo** el entrenamiento y `roc_auc_score(y_test, ...)` mide el AUC en el test **por primera y única vez** (`auc_test`); `auc_cv_candidato` es su AUC de validación cruzada.
- **Qué debe salir:**
    - `candidato: RegresionLogistica · AUC validación 0.755 · AUC test 0.745` y `✔ Ejercicio 5 correcto`.
- **Cómo leerlo:**
    - El AUC en test (0,745) es parecido al de validación cruzada (0,755): resultado **realista y consistente**. Si el test saliera muy por debajo, habría sobreajuste o fuga.
- **Si falla:**
    - `El AUC en test debería ser realista (entre 0,6 y 0,85)…` → no has excluido el modelo con fuga, o no has evaluado en `X_test`.
    - `Test y validación cruzada deberían ser parecidos…` → has reentrenado con otro conjunto o con el modelo equivocado.
    - `NameError: name 'tabla' is not defined`/`'crear_pipeline'` → faltan los ejemplos o ejercicios anteriores.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué candidato se elige y qué AUC tiene en validación y en test?* → **RegresionLogistica**, con **0,755** en validación y **0,745** en test.
2. *¿Por qué se excluye el modelo con fuga?* → Porque su AUC es **irreal** y en producción no se podría usar.
3. *¿Qué pasaría si se elige el modelo mirando el test varias veces?* → El test **dejaría de ser independiente** y la estimación saldría demasiado optimista.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
tabla = pd.read_csv(RUTA_REGISTRO)
legitimos = tabla[~tabla["modelo"].str.contains("Fuga")]
fila = legitimos.loc[legitimos["valor_cv"].idxmax()]
hp = json.loads(fila["hiperparametros"])
if fila["modelo"] == "RegresionLogistica":
    estimador = LogisticRegression(max_iter=2000)
elif fila["modelo"] == "HistGradientBoosting":
    estimador = HistGradientBoostingClassifier(random_state=SEMILLA)
else:
    estimador = RandomForestClassifier(random_state=SEMILLA, n_jobs=2, **{k: v for k, v in hp.items()})
final = crear_pipeline(estimador, X_ent.columns).fit(X_ent, y_ent)
auc_test = roc_auc_score(y_test, final.predict_proba(X_test)[:, 1])
auc_cv_candidato = fila["valor_cv"]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 7. Opcional: MLflow en local

#### A11 · Ejemplo — MLflow en local (opcional, requiere internet)

**MLflow** es una herramienta profesional de seguimiento: guarda parámetros, métricas y modelos y ofrece
una interfaz web (`mlflow ui`). **No es obligatoria** ni necesaria para el proyecto: el CSV cumple con
la rúbrica. Si tu equipo quiere probarla, pon `USAR_MLFLOW = True`. Necesita `%pip install -q mlflow`
(descarga, requiere internet) y **no se ejecuta por defecto**.

- **Qué hace:**
    - `USAR_MLFLOW = False` desactiva la parte opcional; pásalo a `True` solo si tu equipo quiere probar MLflow (necesita instalarlo, con internet).
    - Con `True`, abre un experimento llamado `turisdata_cancelaciones`, registra los parámetros, las métricas (`auc_cv`, `auc_test`) y la huella de los datos, e indica cómo ver el resultado con `mlflow ui`.
- **Qué debe salir:**
    - Con `False` (lo normal): `MLflow desactivado (opcional). El registro en CSV ya cumple lo que pide la rúbrica.`
- **Cómo leerlo:**
    - MLflow es una herramienta profesional, **no obligatoria**: el CSV cumple con la rúbrica.
- **Si falla:**
    - Con `True` y sin MLflow instalado: `ModuleNotFoundError` → instálalo (`pip install mlflow`) o déjalo en `False`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Es obligatorio usar MLflow en el proyecto?* → **No**: el registro en CSV cumple lo que pide la rúbrica.
2. *¿Qué ventaja tiene MLflow frente al CSV?* → Guarda también **modelos y gráficos** y ofrece una **interfaz web** (`mlflow ui`).

</details>

## Mini-reto para tu proyecto · 🏠 en casa

Aplica esto a **tu proyecto**: en el cuaderno común del equipo (`03_modelos.ipynb`), copia `fijar_semillas`,
`registrar` y `experimento` en las primeras celdas, registra ≥ 3 modelos + baseline con tus datos en
`registro.csv` y haz un *commit* con el CSV. Cada integrante debe tener **al menos una fila** con su nombre
en `responsable` (sirve como evidencia de contribución).

*(Este mini-reto se hace en el cuaderno de tu equipo, no aquí: no se autocorrige.)*

## Preguntas de reflexión
1. ¿Qué dos cosas necesita otra persona para reproducir exactamente tu mejor resultado?
2. Dos modelos tienen AUC 0,74 ± 0,02 y 0,75 ± 0,02. ¿Puedes afirmar que el segundo es mejor? ¿Qué elegirías?
3. ¿Por qué no se debe elegir el modelo mirando el resultado en el test? ¿Qué pasa si se hace varias veces?
4. Tu equipo obtiene AUC 0,97 en la primera prueba. Enumera tres cosas que revisarías antes de celebrarlo.

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. La **semilla** fijada (con `fijar_semillas`) y las **versiones** de las librerías (la ficha del entorno); además, los **mismos datos**, que se comprueban con la huella (`DATOS_HASH`).
2. **No**: la diferencia (0,01) es menor que la desviación (0,02), así que es ruido. Elegiría el **más simple, más rápido o más fácil de explicar**, y lo anotaría en la columna `notas`.
3. Porque el test debe ser una estimación **independiente** de cómo funcionará el modelo con datos nuevos. Si se elige mirando el test, el modelo se ajusta (indirectamente) a él y su resultado sale demasiado optimista; si se hace varias veces, es como entrenar con el test.
4. Por ejemplo: **fuga de datos** (variables que no se conocerían al decidir), que el preprocesado esté **dentro de un `Pipeline`** y se ajuste solo con el entrenamiento, que no haya **duplicados** entre entrenamiento y test, y que el resultado se compare con el **baseline** y con la validación cruzada.

</details>

## Resumen: qué has aprendido
- Fijar semillas, anotar versiones y huella de datos.
- Registrar experimentos en un CSV versionado en Git.
- Comparar con validación cruzada (media ± desviación) y reservar el test para el final.
- Reconocer y prevenir fugas de datos.